In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from os import path
path="/content/drive/MyDrive/MachineLearningPraktikum/Praktikum10/data"

In [10]:
df = pd.read_excel(path + "/Praktikum1.xlsx")
df

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,No,Suhu,Angin,Kelas,Selisih Suhu,Selisih Angin,Jarak
0,NaN,NaN,NaN,NaN,NaN,1,10,0,Dingin,36.0,9.0,6.708204
1,NaN,NaN,NaN,NaN,NaN,2,25,0,Panas,81.0,9.0,9.486833
2,NaN,NaN,NaN,NaN,NaN,3,15,5,Dingin,1.0,4.0,2.236068
3,NaN,NaN,NaN,NaN,NaN,4,20,3,Panas,16.0,0.0,4.000000
4,NaN,NaN,NaN,NaN,NaN,5,18,7,Dingin,4.0,16.0,4.472136
5,NaN,NaN,NaN,NaN,NaN,6,20,10,Dingin,16.0,49.0,8.062258
6,NaN,NaN,NaN,NaN,NaN,7,22,5,Panas,36.0,4.0,6.324555
7,NaN,NaN,NaN,NaN,NaN,8,24,6,Panas,64.0,9.0,8.544004
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,Data Testing,Column1,K,4,5.0,NaN,NaN


In [14]:
df_temp = pd.read_excel(path + "/Praktikum1.xlsx", sheet_name="Sheet1", header=0)
df_train = df_temp.iloc[:, 5:9].head(8).copy()
df_train.columns = ['No', 'Suhu', 'Angin', 'Kelas']

In [15]:
# Konversi kolom ke tipe numerik
df_train['No'] = pd.to_numeric(df_train['No'])
df_train['Suhu'] = pd.to_numeric(df_train['Suhu'])
df_train['Angin'] = pd.to_numeric(df_train['Angin'])

In [16]:
# Data Uji untuk Marry
test_suhu = 16
test_angin = 3

In [19]:
from collections import Counter

# --- Definisi Fungsi KNN dan LOOCV ---

def knn_predict(X_train, y_train, X_test, k):
    """Memprediksi kelas untuk satu data uji menggunakan KNN."""
    # Hitung Jarak Euclidean
    distances = np.sqrt(np.sum((X_train - X_test)**2, axis=1))

    # Gabungkan jarak dan kelas
    df_temp = pd.DataFrame({'Jarak': distances, 'Kelas': y_train})
    df_temp_sorted = df_temp.sort_values(by='Jarak')

    # Ambil K tetangga terdekat dan tentukan kelas mayoritas
    top_k = df_temp_sorted.head(k)
    class_counts = Counter(top_k['Kelas'])
    return class_counts.most_common(1)[0][0]

def loocv_accuracy(df_data, k):
    """Menghitung akurasi dengan Leave-One-Out Cross-Validation."""
    X = df_data[['Suhu', 'Angin']].values
    y = df_data['Kelas'].values
    n = len(df_data)
    correct_predictions = 0

    for i in range(n):
        # Latih model dengan N-1 data, uji dengan data ke-i
        X_train = np.delete(X, i, axis=0)
        y_train = np.delete(y, i)
        X_test = X[i]
        y_test = y[i]

        predicted_class = knn_predict(X_train, y_train, X_test, k)

        if predicted_class == y_test:
            correct_predictions += 1

    return correct_predictions / n

In [20]:
# --- Menentukan K Optimal ---

k_values = range(1, len(df_train)) # K=1 sampai K=7
accuracy_scores = {}

for k in k_values:
    accuracy = loocv_accuracy(df_train, k)
    accuracy_scores[k] = accuracy

optimal_k = max(accuracy_scores, key=accuracy_scores.get)

In [21]:
# --- Klasifikasi Marry Menggunakan K Optimal ---

X_test_marry = np.array([test_suhu, test_angin])
X_train_data = df_train[['Suhu', 'Angin']].values
y_train_data = df_train['Kelas'].values

marry_prediction_optimal_k = knn_predict(X_train_data, y_train_data, X_test_marry, optimal_k)

print(f"Persepsi Marry (menggunakan K optimal={optimal_k}): {marry_prediction_optimal_k}")

Persepsi Marry (menggunakan K optimal=1): Dingin
